# Poisson GLM: claim frequency

**Goal:** an interpretable model of the expected number of claims per policy, scored on exactly the same folds and metric as the baseline and the gradient boosting model.

**The model.** `frequency = exp(b0 + b1*x1 + b2*x2 + ...)`. The exponential keeps predictions positive, and every coefficient acts as a **multiplicative factor**: `exp(b)` is how much the frequency is multiplied when a level applies (or when a numeric feature increases by one unit). Claim counts are assumed Poisson with mean `exposure x frequency`.

**Exposure.** The target is the frequency `ClaimNb / Exposure` and `Exposure` is the **sample weight**. This is equivalent to modelling claim counts with an exposure offset (`log(exposure)` added to the linear predictor with coefficient fixed at 1). `expected claims = prediction x exposure`.

**Ridge penalty `alpha`.** A small L2 penalty shrinks noisy coefficients (thin levels) and makes the solution unique even though one-hot columns are redundant with the intercept. Larger alpha = more shrinkage; it is tuned in Step 3.

**Rules (same as the other models):** training set only; frozen grouped folds; Poisson deviance; calibration by exposure band; everything that learns from data lives inside the pipeline and is fitted on a training part only. Features come from `src/features.py` (bands, logs, bonus-malus flags, one-hot with rare levels merged, scaling).

**Plan:** 1. one fold and a look inside the model, 2. cross-validation against the references, 3. tuning alpha, 4. save as a module and check it, 5. a first look against the boosting model.

In [1]:
# Cell 1: setup, training data, references
# (Magics must stay alone on their line: no comments after them.)
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")      # lets the notebook import our project code from src/
import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline

from src.evaluation import (load_train, poisson_deviance, cross_validate, calibration_by_exposure,
                            exposure_band_reference, N_FOLDS, PROCESSED)
from src.features import add_features, make_preprocessor      # row-wise features + learned preprocessing (scaler, one-hot)

train = load_train()
print(train.shape)                       # your earlier run: (541432, 23)

def baseline(tr, va):
    """The reference: one constant rate learned on the training part, scaled by exposure."""
    return (va["Exposure"] * tr["ClaimNb"].sum() / tr["Exposure"].sum()).to_numpy()

(541432, 23)


## Step 1: one fold, one model, a look inside
Fold 0 is the validation part. The pipeline is `add_features` (fixed rules) -> `make_preprocessor` (scaler + one-hot, fitted on the training part only) -> `PoissonRegressor`.

In [2]:
# Cell 2: define the model and fit it on fold 0
def make_glm(alpha=1e-4):
    return Pipeline([
        ("pre", make_preprocessor()),                              # learns scaler statistics and category levels
        ("glm", PoissonRegressor(alpha=alpha, max_iter=1000)),     # log link, Poisson deviance, ridge penalty alpha
    ])

def fit_glm(tr, alpha=1e-4):
    model = make_glm(alpha)
    # target = frequency, weight = exposure: equivalent to claim counts with an exposure offset
    model.fit(add_features(tr), tr["ClaimNb"] / tr["Exposure"], glm__sample_weight=tr["Exposure"])
    return model

def expected(model, d):
    """Expected claims = predicted frequency x exposure."""
    return model.predict(add_features(d)) * d["Exposure"].to_numpy()

tr, va = train[train["Fold"] != 0], train[train["Fold"] == 0]
glm = fit_glm(tr)
print("solver iterations:", glm.named_steps["glm"].n_iter_, "(limit 1000)")      # below the limit = converged

mu_tr, mu_va = expected(glm, tr), expected(glm, va)
d_tr = poisson_deviance(tr["ClaimNb"], mu_tr)
d_va = poisson_deviance(va["ClaimNb"], mu_va)
d_base = poisson_deviance(va["ClaimNb"], baseline(tr, va))
assert np.isclose(d_base, 0.3291, atol=1e-4)                 # fold-0 baseline from the reference run
print(f"deviance  train: {d_tr:.4f} | validation: {d_va:.4f} | baseline on the same fold: {d_base:.4f}")
print(f"improvement on validation: {d_base - d_va:.4f} ({1 - d_va / d_base:.1%})")
print("validation actual/predicted claims:", round(va["ClaimNb"].sum() / mu_va.sum(), 4))

# How to read it: fewer iterations than the limit = the solver converged (otherwise a ConvergenceWarning appears).
# A validation score clearly below the baseline's 0.3291 = the features carry real signal.
# A small train/validation gap = no overfitting (65 coefficients on 433,000 rows).
# Hypothesis, not yet run: a clear gain over the baseline, somewhat smaller than the boosting model's 0.0264 on this fold.

solver iterations: 24 (limit 1000)
deviance  train: 0.3123 | validation: 0.3097 | baseline on the same fold: 0.3291
improvement on validation: 0.0194 (5.9%)
validation actual/predicted claims: 0.9999


In [3]:
# Cell 3: the coefficient table = what the model learned, as multiplicative factors
def coefficient_table(model, tr):
    """Categories: relativity vs the exposure-weighted average level of the variable.
    Scaled numerics: factor per +1 standard deviation. 0/1 flags: factor when the flag is on."""
    pre, glm_ = model.named_steps["pre"], model.named_steps["glm"]
    X = pre.transform(add_features(tr))
    e = tr["Exposure"].to_numpy()
    share = np.asarray(X.T.dot(e)).ravel() / e.sum()          # exposure share of every column (sums to 1 per variable)
    rows = []
    for name, b, s in zip(pre.get_feature_names_out(), glm_.coef_, share):
        if name.startswith("cat__"):
            var, level = name[5:].split("_", 1)
            rows.append((var, level, b, s))
        else:
            rows.append((name.split("__", 1)[1], "", b, np.nan))
    t = pd.DataFrame(rows, columns=["variable", "level", "coef", "exposure_share"])
    is_cat = t["exposure_share"].notna()
    t["wcoef"] = t["coef"] * t["exposure_share"]
    avg = t[is_cat].groupby("variable")["wcoef"].sum()         # exposure-weighted average coefficient per variable
    t["avg"] = t["variable"].map(avg).fillna(0.0)
    t["relativity"] = np.exp(t["coef"] - t["avg"]).round(3)    # only DIFFERENCES between levels of a variable are meaningful
    return t[["variable", "level", "coef", "exposure_share", "relativity"]].round(4)

ct = coefficient_table(glm, tr)
print(ct.to_string(index=False))

# Sanity checks against the EDA (printed, not asserted: a False is a finding to investigate, not a bug by itself)
rel = ct.set_index(["variable", "level"])["relativity"]
checks = {
    "18-21 riskier than 31-40":              rel[("DrivAgeBand", "18-21")] > rel[("DrivAgeBand", "31-40")],
    "age-0 vehicle riskier than 6-10 years": rel[("VehAgeBand", "00")] > rel[("VehAgeBand", "06-10")],
    "bm_off_ladder above 1":                 rel[("bm_off_ladder", "")] > 1,
    "bm_is_100 above 1":                     rel[("bm_is_100", "")] > 1,
    "logBonusMalus above 1":                 rel[("logBonusMalus", "")] > 1,
    "logDensity above 1":                    rel[("logDensity", "")] > 1,
}
print(checks)

# Reading guide: relativities of a categorical variable are relative to its exposure-weighted AVERAGE level.
# Hypothesis: brand B12, whose raw EDA relativity was 1.35, should shrink toward 1 here because the new-vehicle effect
# is now carried by VehAgeBand. Compare with the raw EDA table to see how much of the 'brand effect' was a new-car effect.

     variable                       level    coef  exposure_share  relativity
logBonusMalus                              0.2844             NaN       1.329
   logDensity                              0.0830             NaN       1.087
     bm_is_50                              0.0918             NaN       1.096
    bm_is_100                              0.3842             NaN       1.468
bm_off_ladder                              0.6814             NaN       1.977
  DrivAgeBand                       18-21  0.1816          0.0125       1.233
  DrivAgeBand                       22-25 -0.0404          0.0328       0.988
  DrivAgeBand                       26-30 -0.2256          0.0816       0.821
  DrivAgeBand                       31-40 -0.1848          0.2343       0.855
  DrivAgeBand                       41-50  0.0871          0.2453       1.122
  DrivAgeBand                       51-60  0.0249          0.2110       1.054
  DrivAgeBand                       61-70 -0.0056          0.106

## Step 2: cross-validation against the references
`cross_validate` fits the pipeline on four folds and predicts the fifth, five times. All models go through this same loop, so the scores are comparable fold by fold.

In [4]:
# Cell 4: full CV, paired comparison, calibration, out-of-fold predictions
ALPHA = 1e-4

def glm_fit_predict(tr, va, alpha=ALPHA):
    return expected(fit_glm(tr, alpha), va)

oof_glm, s_glm = cross_validate(train, glm_fit_predict)         # five fits (a few minutes); also cross-checks the metric
_, s_b = cross_validate(train, baseline)
_, s_e = cross_validate(train, exposure_band_reference)
assert np.isclose(s_b.mean(), 0.3312, atol=1e-4)                 # the shared baseline still reproduces the reference

print("baseline          :", s_b.round(4), "| mean", round(s_b.mean(), 4))
print("exposure yardstick:", s_e.round(4), "| mean", round(s_e.mean(), 4))
print("GLM               :", s_glm.round(4), "| mean", round(s_glm.mean(), 4), "+/-", round(s_glm.std(), 4))
print("improvement vs baseline  per fold:", (s_b - s_glm).round(4), "| mean", round((s_b - s_glm).mean(), 4))
print("improvement vs yardstick per fold:", (s_e - s_glm).round(4), "| mean", round((s_e - s_glm).mean(), 4))
print("\noverall actual/predicted:", round(train["ClaimNb"].sum() / oof_glm.sum(), 4), "\n")
print(calibration_by_exposure(train, oof_glm))

# Out-of-fold expected claims per policy: the input of the final comparison with the boosting model
pd.DataFrame({"IDpol": train["IDpol"], "pred_claims": oof_glm}).to_csv(PROCESSED / "oof_glm.csv", index=False)

# References: baseline 0.3312, yardstick 0.3183, boosting about 0.3044 (raw features) / 0.3037 (final, expected).
# Hypotheses, not yet run: positive gains over the baseline in EVERY fold, a mean deviance close to (probably a little above)
# the boosting model's, and a calibration by exposure with the same structural pattern: the model cannot see exposure.

baseline          : [0.3291 0.3305 0.333  0.3349 0.3285] | mean 0.3312
exposure yardstick: [0.3165 0.318  0.3199 0.3215 0.3153] | mean 0.3183
GLM               : [0.3097 0.3118 0.3123 0.3174 0.3091] | mean 0.3121 +/- 0.0029
improvement vs baseline  per fold: [0.0194 0.0187 0.0207 0.0175 0.0194] | mean 0.0191
improvement vs yardstick per fold: [0.0068 0.0062 0.0076 0.0041 0.0062] | mean 0.0062

overall actual/predicted: 1.0002 

             policies  actual  predicted  actual/predicted
Exposure                                                  
(0.0, 0.1]     102054    2785      802.5             3.470
(0.1, 0.25]     76054    2890     1733.0             1.668
(0.25, 0.5]    105108    5212     4953.8             1.052
(0.5, 0.75]     73967    4822     5479.7             0.880
(0.75, 1.0]    184249   13123    15857.3             0.828


## Step 3: tuning the ridge penalty
`alpha` is the only hyperparameter. A change is **adopted only if it is better in at least 4 of 5 folds and the mean gain is at least about 0.0005** (the paired gap between two reference models varied by only about 0.0004 across folds). If nothing qualifies, we keep the default and say that alpha does not matter here.

In [5]:
# Cell 5: alpha grid (25 fits: allow time). Very small alpha = almost unpenalized.
ALPHAS = [1e-8, 1e-6, 1e-5, 1e-4, 1e-3]

scores_alpha = {}
for a in ALPHAS:
    _, scores_alpha[a] = cross_validate(train, lambda tr, va, a=a: glm_fit_predict(tr, va, alpha=a), check=False)
    print(f"alpha={a:g} done")

s_ref = scores_alpha[ALPHA]
rows = []
for a, s in scores_alpha.items():
    rows.append({"alpha": a, "mean_deviance": s.mean(), "gain_vs_1e-4": (s_ref - s).mean(),
                 "folds_better": int((s < s_ref).sum()), "worst_fold_gain": (s_ref - s).min()})
print(pd.DataFrame(rows).round(5).to_string(index=False))

# Reading: with 541,000 rows and merged rare levels, the penalty only matters for the thinnest levels, so I expect
# differences below the 0.0005 threshold (hypothesis). If no alpha qualifies, keep 1e-4. If a very small alpha is the
# best, check that the solver still converged (n_iter_ below the limit) before trusting it.

alpha=1e-08 done
alpha=1e-06 done
alpha=1e-05 done
alpha=0.0001 done
alpha=0.001 done
  alpha  mean_deviance  gain_vs_1e-4  folds_better  worst_fold_gain
0.00000        0.31209      -0.00002             1         -0.00005
0.00000        0.31209      -0.00002             1         -0.00004
0.00001        0.31208      -0.00001             2         -0.00004
0.00010        0.31207       0.00000             0          0.00000
0.00100        0.31215      -0.00008             1         -0.00020


## Step 4: save the model as a module and check it
`src/glm.py` holds the same pipeline, so the comparison notebook and the final test evaluation import the exact code used here. The next cell proves the module and the notebook give identical predictions.

In [6]:
# Cell 6: src/glm.py must reproduce the notebook (compare on fold 0)
from src.glm import fit_predict as fit_predict_module

pred_nb = glm_fit_predict(tr, va)                 # notebook version
pred_mod = fit_predict_module(tr, va)             # module version
assert np.allclose(pred_nb, pred_mod), "the module and the notebook differ"
print("module and notebook agree on fold 0; mean predicted claims:", round(pred_mod.mean(), 5))

module and notebook agree on fold 0; mean predicted claims: 0.05316


## Step 5: a first look against the boosting model
Both models were scored on the same folds, so they can be compared fold by fold. This is only a preview: the proper comparison (paired scores, calibration by exposure band, a lift curve, and the one-time test evaluation) comes in its own notebook.

In [7]:
# Cell 7: preview of GLM vs boosting, from the saved out-of-fold predictions
masks = [(train["Fold"] == f).to_numpy() for f in range(N_FOLDS)]
oof_gbm = pd.read_csv(PROCESSED / "oof_gbm.csv")
assert (oof_gbm["IDpol"].to_numpy() == train["IDpol"].to_numpy()).all()       # same policies, same order
pred_gbm = oof_gbm["pred_claims"].to_numpy()
s_gbm = np.array([poisson_deviance(train.loc[m, "ClaimNb"], pred_gbm[m]) for m in masks])

print("boosting file :", s_gbm.round(4), "| mean", round(s_gbm.mean(), 4))     # 0.3044 = raw-feature version, ~0.3037 = final version
print("GLM           :", s_glm.round(4), "| mean", round(s_glm.mean(), 4))
gap = s_glm - s_gbm
print("GLM minus boosting per fold:", gap.round(4), "| mean", round(gap.mean(), 4), "| folds where boosting is better:", int((gap > 0).sum()))
# Positive = the boosting model is better. A gain is only trusted if it is consistent across folds and above about 0.0005.

boosting file : [0.302  0.3038 0.3026 0.3095 0.3011] | mean 0.3038
GLM           : [0.3097 0.3118 0.3123 0.3174 0.3091] | mean 0.3121
GLM minus boosting per fold: [0.0077 0.008  0.0098 0.008  0.008 ] | mean 0.0083 | folds where boosting is better: 5
